In [1]:
import pyvista as pv
import gstools as gs
import numpy as np
from astropy.io import fits

In [13]:
pv.set_jupyter_backend("trame")
#pv.set_jupyter_backend("client")

In [14]:
N = 256
dim, spacing = (N, N, N), (1,1,1)
grid = pv.ImageData(dimensions=dim, spacing=spacing)

In [15]:
field_synth = 'velocity'
model_gs    = 'stable'
alpha       = 1.5
len_scale   = [16, 16, 16]
angles      = [0.8, 0.8, 0.8]
seed        = 19880127

In [16]:
model = gs.Stable(dim=3, len_scale=len_scale, angles=angles)
srf = gs.SRF(model, seed=seed)

In [17]:
field = srf.mesh

In [18]:
srf.mesh(grid, points='points', name='random-field')

array([-1.27964498, -1.28407739, -1.47883468, ..., -0.76875301,
       -0.816935  , -0.98914161], shape=(16777216,))

In [19]:
grid.contour(isosurfaces=6).plot(smooth_shading=True, use_transparency=True, opacity=0.3, cmap='coolwarm')

Widget(value='<iframe src="http://localhost:61825/index.html?ui=P_0x197ebaf0350_3&reconnect=auto" class="pyvis…

In [11]:
import os
import sys
import pyvista as pv

print("Process ID:", os.getpid())
print("Python:", sys.executable)
print("PyVista:", pv.__version__)
print("Backend:", pv.global_theme.jupyter_backend)

Process ID: 19136
Python: C:\Users\javas\anaconda3\envs\gstools\python.exe
PyVista: 0.48.4
Backend: client


In [12]:
print(
    "Extension available:",
    pv.global_theme.trame.jupyter_extension_available,
)
print(
    "Extension enabled:",
    pv.global_theme.trame.jupyter_extension_enabled,
)

Extension available: False
Extension enabled: False


In [9]:
print(type(srf.field))

<class 'numpy.ndarray'>


In [10]:
print(srf.field.shape)

(16777216,)


In [11]:
field_3d = np.asarray(grid.point_data['random-field']).reshape(dim, order='F')

In [12]:
print(type(field_3d))
print(field_3d.shape)
print(field_3d.min())
print(field_3d.max())
print(field_3d.mean())
print(field_3d.std())

<class 'numpy.ndarray'>
(256, 256, 256)
-4.292760396036961
4.267679779764338
-0.023248303695602768
0.9614111693986683


In [13]:
header = fits.Header()

header["MODEL"] = model_gs
header["SEED"]  = seed
header["M3D"]   = alpha

header["LSCALE_X"] = len_scale[0]
header["LSCALE_Y"] = len_scale[1]
header["LSCALE_Z"] = len_scale[2]

header["ANGLE_X"] = angles[0]
header["ANGLE_Y"] = angles[1]
header["ANGLE_Z"] = angles[2]

header["DX"] = spacing[0]
header["DY"] = spacing[1]
header["DZ"] = spacing[2]


In [14]:
name = model_gs + "_3d_" + field_synth 
print(name)

stable_3d_velocity


In [16]:
fits.writeto(
    name + ".fits",
    field_3d.astype("float32"),
    header=header,
    overwrite=True,
)